In [75]:
# Import necessary libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

import swcol as sw
parse_tech, colors, tech_order, tech_colors = sw.template.get() # Template

In [76]:
model_inputs_path = '../../model/inputs/'
model_outputs_path = '../../model/outputs/'

In [77]:

gen_build_costs=pd.read_csv(model_inputs_path+'gen_build_costs.csv')
gen_build_costs = gen_build_costs[gen_build_costs['build_year'] <= 2023]

gen_info=pd.read_csv(model_inputs_path+'gen_info.csv')
gen_build_costs = pd.merge(gen_build_costs, gen_info, on='GENERATION_PROJECT')

gen_build_costs['gen_tech'] = gen_build_costs['gen_tech'].replace(parse_tech)
gen_costs = gen_build_costs[['gen_tech','gen_overnight_cost','gen_fixed_om','gen_variable_om']]
print('Shape',gen_costs.shape)
gen_costs.sample(5)

Shape (241, 4)


,gen_tech,gen_overnight_cost,gen_fixed_om,gen_variable_om
28,Thermal,1406000,143220,9.30
221,Solar,1327000,15970,1.00
79,Hydro,3083000,43780,1.39
210,Solar,1327000,15970,1.00
209,Solar,1327000,15970,1.00


In [78]:
gen_overnight_cost  = gen_costs.copy()
gen_overnight_cost = gen_overnight_cost[['gen_tech','gen_overnight_cost']]

gen_overnight_cost = gen_overnight_cost.groupby(['gen_tech']).agg({'gen_overnight_cost':'mean'}).reset_index()

fig = px.bar(
    gen_overnight_cost, x='gen_tech', y='gen_overnight_cost', 
    color_discrete_sequence=[colors[0]],
    labels={'gen_tech': 'Generation Technology', 'gen_overnight_cost': 'Mean Overnight Cost [MUSD/MW]'},
    height=9*50, width=16*50, category_orders={"gen_tech": tech_order},
    template='plotly_white')
fig.write_image(f"../images/Mean Overnight Cost.png")
fig.show()

In [79]:
gen_fixed_om  = gen_costs.copy()
gen_fixed_om = gen_fixed_om[['gen_tech','gen_fixed_om']]

gen_fixed_om = gen_fixed_om.groupby(['gen_tech']).agg({'gen_fixed_om':'mean'}).reset_index()

fig = px.bar(
    gen_fixed_om, x='gen_tech', y='gen_fixed_om',
    color_discrete_sequence=[colors[0]],
    labels={'gen_tech': 'Generation Technology', 'gen_fixed_om': 'Mean Fixed O&M Costs [USD/MW year]'},
    height=9*50, width=16*50, category_orders={"gen_tech": tech_order},
    template='plotly_white')

fig.write_image(f"../images/Mean Fixed O&M Costs.png")
fig.show()

In [80]:
gen_variable_om  = gen_costs.copy()
gen_variable_om = gen_variable_om[['gen_tech','gen_variable_om']]

gen_variable_om = gen_variable_om.groupby(['gen_tech']).agg({'gen_variable_om':'mean'}).reset_index()

fig = px.bar(
    gen_variable_om, x='gen_tech', y='gen_variable_om',
    color_discrete_sequence=[colors[0]],
    labels={'gen_tech': 'Generation Technology', 'gen_variable_om': 'Mean Variable O&M Costs [USD/MWh]'},
    height=9*50, width=16*50, category_orders={"gen_tech": tech_order},
    template='plotly_white')

fig.write_image(f"../images/Mean Variable O&M Costs.png")
fig.show()

In [81]:
import pandas as pd
import plotly.graph_objects as go
# Merge DataFrames
df = gen_overnight_cost.merge(gen_fixed_om, on='gen_tech').merge(gen_variable_om, on='gen_tech')
df['gen_overnight_cost'] = df['gen_overnight_cost'] / 1000000
df['gen_fixed_om'] = df['gen_fixed_om'] / 1000
# Create the table
header = ['<b>Generation Technology</b>', '<b>Mean Overnight Cost [MUSD/MW]</b>', '<b>Mean Fixed O&M Costs [USD/MW year]</b>', '<b>Mean Variable O&M Costs [USD/MWh]</b>']
cells = [
    df['gen_tech'].tolist(),
    df['gen_overnight_cost'].round(2).tolist(),
    df['gen_fixed_om'].round(2).tolist(),
    df['gen_variable_om'].round(4).tolist()
]

fig = go.Figure(data=[go.Table(
    header=dict(
        values=header,
        fill_color='lightgray',
        align='left', line_color='black'
    ),
    cells=dict(
        values=cells,
        fill_color='white',
        align='left', line_color='black'
    )
)])

fig.update_layout(title='Costs by Technology', height=9*50, width=16*50, template="plotly_white")
fig.write_image("../images/Costs by Technology.png")
fig.show()

In [82]:
fuel_cost=pd.read_csv(model_inputs_path+'fuel_cost.csv')

# Calcular el promedio del costo por tipo de combustible
fuel_cost = fuel_cost.groupby('fuel')['fuel_cost'].mean().reset_index()
fuel_cost['fuel'] = fuel_cost['fuel'].replace(
    {
        'ACPM': 'Diesel',
        'CARBON': 'Coal',
        'COMBUSTOLEO': 'Fuel Oil',
        'GASIMPOR': 'Imported Gas',
        'GASNACIO': 'National Gas'
    })

fig = px.bar(
    fuel_cost, x='fuel', y='fuel_cost',
    color_discrete_sequence=[colors[0]],
    labels={'fuel': 'Fuel', 'fuel_cost': 'Fuel Cost [USD/MMBtu]'},
    height=9*50, width=16*50, category_orders={"gen_tech": tech_order},
    template='plotly_white')
fig.write_image("../images/Fuel Cost.png")
fig.show()

In [83]:
model_path = '../../../model/inputs/'
fuels = pd.read_csv(model_inputs_path+'fuels.csv')
fuels = fuels[['fuel','co2_intensity']]

# Lista de valores de interés
filtered_fuels = ['GASIMPOR', 'GASNACIO', 'ACPM', 'CARBON', 'COMBUSTOLEO', 'GLP']

# Filtra el DataFrame donde los valores de 'columna_deseada' están en la lista de interés
fuels = fuels[fuels['fuel'].isin(filtered_fuels)]
fuels['fuel'] = fuels['fuel'].str.replace('GASNACIO', 'National Gas')
fuels['fuel'] = fuels['fuel'].str.replace('GASIMPOR', 'Imported Gas')
fuels['fuel'] = fuels['fuel'].str.replace('ACPM', 'Diesel')
fuels['fuel'] = fuels['fuel'].str.replace('CARBON', 'Coal')
fuels['fuel'] = fuels['fuel'].str.replace('COMBUSTOLEO', 'Fuel Oil')
fuels['fuel'] = fuels['fuel'].str.replace('GLP', 'LPG')
import plotly.express as px
# Crear la gráfica de barras, usando 'gen_tech' como color
fig = px.bar(
    fuels, x='fuel', y='co2_intensity',
    title='CO2 Intensity per Technology',
    color_discrete_sequence=[colors[0]],
    labels={'co2_intensity': 'CO2 Intensity [tCO2/MMBtu]', 'fuel': 'Fuel Type'},
    height=9*50, width=16*50, category_orders={"gen_tech": tech_order},
    template='plotly_white')
fig.write_image("../images/CO2 Intensity.png")
# Mostrar la gráfica
fig.show()

In [84]:
import pandas as pd
import plotly.graph_objects as go
# Merge DataFrames
df = fuels.merge(fuel_cost, on='fuel', how='outer')
df = df.fillna(0)
# Create the table
header = ['<b>Generation Technology</b>', '<b>Fuel Cost [USD/MMBtu]</b>', '<b>CO2 Intensity [tCO2/MMBtu year]</b>']
cells = [
    df['fuel'].tolist(),
    df['fuel_cost'].round(2).tolist(),
    df['co2_intensity'].round(2).tolist()
]

fig = go.Figure(data=[go.Table(
    header=dict(
        values=header,
        fill_color='lightgray',
        align='left', line_color='black'
    ),
    cells=dict(
        values=cells,
        fill_color='white',
        align='left', line_color='black'
    )
)])

fig.update_layout(title='Fuels', height=9*50, width=16*50, template="plotly_white")
fig.write_image("../images/Fuels.png")
fig.show()